***1. Exploratory analysis of data and visualizations***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

In [ ]:
df = pd.read_csv("train.csv")

df.head()
df.info()

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.dropna()

In [ ]:
df.describe()

In [ ]:
df["length"] = df["Context"].apply(len)

plt.hist(df["length"], bins=50)
plt.title("Distribution of Patient Message Length")
plt.xlabel("Message Length")
plt.ylabel("Frequency")
plt.show()

In [ ]:
text = " ".join(df["Context"])

wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)

plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Frequent Words in Patient Messages")
plt.show()

***2. Data preprocessing***

In [ ]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^\w\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [ ]:
df["clean_text"] = df["Context"].apply(clean_text)

print(df[["Context","clean_text"]].head())

***3. Training in the emotional classification model***

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["clean_text"])

In [ ]:
emotion_patterns = {
    'anxiety': ['anxiety', 'anxious', 'worry', 'worried', 'nervous', 'panic', 'fear', 'scared', 'tense', 'stress', 'overwhelm', 'dread'],
    'sadness': ['sad', 'depressed', 'unhappy', 'grief', 'sorrow', 'down', 'cry', 'hopeless', 'lonely', 'hurt', 'pain', 'alone'],
    'angry': ['angry', 'anger', 'frustrated', 'mad', 'annoyed', 'rage', 'irritated', 'hate', 'upset', 'furious', 'pissed'],
    'stress': ['stress', 'overwhelmed', 'pressure', 'burnout', 'tension', 'exhausted', 'tired', 'busy', 'work', 'deadline'],
    'happiness': ['happy', 'glad', 'joy', 'excited', 'grateful', 'thankful', 'pleased', 'delighted', 'wonderful', 'great', 'good'],
    'fear': ['fear', 'afraid', 'terrified', 'frightened', 'horror', 'scary', 'terrible', 'awful', 'nightmare'],
    'apathy': ['bored', 'indifferent', 'numb', 'empty', 'nothing', 'whatever', 'pointless', 'meaningless', 'tired', 'whatever'],
    'confusion': ['confused', 'unsure', 'uncertain', 'doubt', 'puzzled', 'lost', 'understand', 'confusing', 'clear'],
    'solitude': ['alone', 'lonely', 'isolated', 'abandoned', 'deserted', 'nobody', 'no one', 'friends', 'social'],
    'hope': ['hope', 'hopeful', 'better', 'improve', 'recover', 'heal', 'progress', 'future', 'positive'],
    'guilt': ['guilt', 'guilty', 'blame', 'regret', 'sorry', 'apologize', 'mistake', 'wrong'],
    'shame': ['shame', 'embarrassed', 'humiliated', 'ashamed', 'awkward', 'stupid', 'foolish']
}

In [ ]:
def detect_emotion(text):
    for emotion, patterns in emotion_patterns.items():
        for pattern in patterns:
            if pattern in text:
                return emotion
    return "neutral"
df["emotion"] = df["clean_text"].apply(detect_emotion)

In [ ]:
y = df["emotion"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

emotion_model = LogisticRegression(max_iter=200)

emotion_model.fit(X_train, y_train)

***4. Model evaluation***

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
y_pred = emotion_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

In [ ]:
!pip install sentence-transformers

***5. Implementation of the complete hybrid system***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
context_embeddings = embedding_model.encode(df["Context"].tolist())

In [ ]:
def retrieve_response(user_input):

    user_embedding = embedding_model.encode([user_input])

    similarity = cosine_similarity(user_embedding, context_embeddings)

    index = similarity.argmax()

    return df.iloc[index]["Response"]

In [ ]:
def mental_health_chatbot():

    user_input = input("Text from the Patient: ")
    clean = clean_text(user_input)
    vector = vectorizer.transform([clean])
    emotion = emotion_model.predict(vector)[0]
    response = retrieve_response(user_input)

    print("\nDetected Emotion:", emotion)
    print("\nEmpathetic Response:", response)
    print("\nThis system is an academic demonstration and does not replace professional mental health support.")

In [ ]:
mental_health_chatbot()


In [ ]:
mental_health_chatbot()

In [ ]:
mental_health_chatbot()

***6. Ethical reflection on the use of AI in mental health***

AI can be a great help when we try to learn about new things, but when it comes to mental health and emotions, AI can pose risks by not fully understanding the emotions a patient is trying to express. Therefore, its use in this setting should be limited to psychologists and under specific parameters.